In [1]:
import os

base_dir = r"D:\processed_slices\train"
total_images = 0

for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.lower().endswith('.png'):
            total_images += 1

print("Total images in training folder:", total_images)


Total images in training folder: 84755


In [2]:
import os

base_dir = r"D:\processed_slices\train"
class_names = ['AD', 'CN', 'MCI']
total_images = 0

for class_name in class_names:
    path = os.path.join(base_dir, class_name, 'axial')
    if os.path.exists(path):
        num_files = len([f for f in os.listdir(path) if f.lower().endswith('.png')])
        print(f"{class_name} axial images: {num_files}")
        total_images += num_files
    else:
        print(f" Path not found: {path}")

print("Total axial images:", total_images)


AD axial images: 17545
CN axial images: 25795
MCI axial images: 41415
Total axial images: 84755


In [ ]:
import os
import numpy as np
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.vgg16 import preprocess_input
from tqdm import tqdm

# Path to the trained model
model_path = "model_axial.h5"
model = load_model(model_path)

# Create feature extractor (last dense layer before output)
feature_extractor = Model(inputs=model.input, outputs=model.layers[-2].output)

# Define base path to training axial slices
base_dir = r"D:\processed_slices\train"
classes = ['AD', 'CN', 'MCI']
label_map = {cls: idx for idx, cls in enumerate(classes)}

# Prepare lists to store features, labels, and image paths
features = []
labels = []
image_paths = []

# Process each image
for cls in classes:
    class_dir = os.path.join(base_dir, cls, 'axial')
    if not os.path.exists(class_dir):
        print(f"⚠️ Directory not found: {class_dir}")
        continue

    for fname in tqdm(os.listdir(class_dir), desc=f"Processing {cls}"):
        if fname.lower().endswith('.png'):
            img_path = os.path.join(class_dir, fname)
            try:
                # Load and preprocess image to 224x224
                img = load_img(img_path, target_size=(224, 224))
                img_array = img_to_array(img)
                img_array = np.expand_dims(img_array, axis=0)
                img_array = preprocess_input(img_array)

                # Extract feature vector
                feat = feature_extractor.predict(img_array, verbose=0)[0]
                features.append(feat)
                labels.append(label_map[cls])
                image_paths.append(img_path)

            except Exception as e:
                print(f" Failed to process {img_path}: {e}")

# Convert to numpy arrays
features = np.array(features)
labels = np.array(labels)
image_paths = np.array(image_paths)

# Save the feature data
np.save("train_features_axial.npy", features)
np.save("train_labels_axial.npy", labels)
np.save("train_image_paths_axial.npy", image_paths)

print(" Feature vectors saved:")
print("train_features_axial.npy")
print("train_labels_axial.npy")
print("train_image_paths_axial.npy")


In [ ]:
import os
import numpy as np

batch_dir = "features_batches_axial"
counts = {'AD': 0, 'CN': 0, 'MCI': 0}
label_map = {0: 'AD', 1: 'CN', 2: 'MCI'}

for file in sorted(os.listdir(batch_dir)):
    if file.startswith("labels_") and file.endswith(".npy"):
        labels = np.load(os.path.join(batch_dir, file))
        for label in labels:
            class_name = label_map[int(label)]
            counts[class_name] += 1

print(" Processed image counts per class:")
for cls in counts:
    print(f"{cls}: {counts[cls]} images extracted")


In [ ]:
import os
import numpy as np

# Directory containing all feature batches
batch_dir = "features_batches_axial"

# Initialize full arrays
all_features, all_labels, all_paths = [], [], []

# Get sorted list of batch files (based on index number)
batch_indices = sorted([
    int(f.split("_")[1].split(".")[0])
    for f in os.listdir(batch_dir) if f.startswith("features_")
])

print(f" Merging {len(batch_indices)} batches...")

# Load and concatenate each batch
for idx in batch_indices:
    f_feat = os.path.join(batch_dir, f"features_{idx}.npy")
    f_lbls = os.path.join(batch_dir, f"labels_{idx}.npy")
    f_paths = os.path.join(batch_dir, f"paths_{idx}.npy")

    features = np.load(f_feat)
    labels = np.load(f_lbls)
    paths = np.load(f_paths)

    all_features.append(features)
    all_labels.append(labels)
    all_paths.append(paths)

#  Stack everything
all_features = np.vstack(all_features)
all_labels = np.concatenate(all_labels)
all_paths = np.concatenate(all_paths)

# Save final merged files
np.save("train_features_axial.npy", all_features)
np.save("train_labels_axial.npy", all_labels)
np.save("train_image_paths_axial.npy", all_paths)

print(" Merge complete!")
print(f" Total feature vectors: {all_features.shape[0]}")
print(f" Feature shape per image: {all_features.shape[1]}")


In [ ]:
import numpy as np

features = np.load("train_features_axial.npy")       # Shape: (84755, 256)
labels = np.load("train_labels_axial.npy")           # Shape: (84755,)
image_paths = np.load("train_image_paths_axial.npy") # Shape: (84755,)


In [ ]:
# now we will combine the features and the labels
import numpy as np
import pandas as pd

# Load the .npy files
features = np.load("train_features_axial.npy")       # Shape: (84755, 256)
labels = np.load("train_labels_axial.npy")           # Shape: (84755,)

# Combine features and labels
combined = np.hstack((features, labels.reshape(-1, 1)))  # Shape: (84755, 257)

# Create a DataFrame
df = pd.DataFrame(combined)

# Optionally name columns
feature_columns = [f"f{i}" for i in range(256)]
df.columns = feature_columns + ["label"]

# Save to CSV
df.to_csv("axial_features_and_labels_only.csv", index=False)

print(" Saved: axial_features_and_labels_only.csv")


In [ ]:
# for text embeddings
import pandas as pd

# Load your file
df = pd.read_csv("ADNI1_Complete_1Yr_1.5T_12_20_2024.csv")

# Preview the first few rows
print(df.shape)
df.head()


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer

# Load CSV
df = pd.read_csv("ADNI1_Complete_1Yr_1.5T_12_20_2024.csv")

# Remove label 
df_cleaned = df.drop(columns=["Group", "Downloaded", "Modality", "Type", "Format"])

# Keep Image ID separately for future mapping
image_ids = df_cleaned["Image Data ID"].values

# Drop ID from the text encoding input
text_only = df_cleaned.drop(columns=["Image Data ID"])

# Convert each row to string and generate sentence embeddings
texts = text_only.astype(str).agg(" ".join, axis=1)

# Generate embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
textual_embeddings = model.encode(texts, show_progress_bar=True)

# Save with Image IDs
import numpy as np
np.save("text_embeddings_cleaned.npy", textual_embeddings)
np.save("text_image_ids.npy", image_ids)


In [ ]:
import numpy as np
import pandas as pd

# Load the cleaned text embeddings and associated image IDs
text_embeddings = np.load("text_embeddings_cleaned.npy")           # shape: (2294, 384)
text_image_ids = np.load("text_image_ids.npy", allow_pickle=True)  # shape: (2294,)

# Convert to DataFrame and add IDs as the first column
df = pd.DataFrame(text_embeddings)
df.insert(0, "Image_ID", text_image_ids)

# Save to CSV
df.to_csv("text_embeddings_cleaned_with_ids.csv", index=False)

print(" Saved: text_embeddings_cleaned_with_ids.csv")
print(" Shape:", df.shape)


In [ ]:
import numpy as np
from tqdm import tqdm
import os

# Load image feature data
image_features = np.load("train_features_axial.npy")       # (84755, 256)
image_labels = np.load("train_labels_axial.npy")           # (84755,)
image_paths = np.load("train_image_paths_axial.npy")       # (84755,)

# Load cleaned textual embeddings and IDs
textual_embeddings = np.load("text_embeddings_cleaned.npy")   # (2294, 384)
text_ids = np.load("text_image_ids.npy", allow_pickle=True)      # (2294,)

# Step 1: Build a lookup from Image ID → text embedding
text_lookup = {id_: emb for id_, emb in zip(text_ids, textual_embeddings)}

# Step 2: Match each image with its text embedding (based on ID prefix)
fused_features = []
fused_labels = []
matched_count = 0

for i, (img_feat, label, path) in enumerate(tqdm(zip(image_features, image_labels, image_paths), total=len(image_paths))):
    filename = os.path.basename(path)  # e.g., 'I31143_AD_axial_55.png'
    img_id = filename.split('_')[0]             # 'I31143'

    if img_id in text_lookup:
        text_feat = text_lookup[img_id]
        fused = np.concatenate([img_feat, text_feat])  # shape (640,)
        fused_features.append(fused)
        fused_labels.append(label)
        matched_count += 1

print(f" Matched samples: {matched_count}")

# Convert to arrays and save
fused_features = np.array(fused_features)
fused_labels = np.array(fused_labels)

np.save("fused_features_clean.npy", fused_features)
np.save("fused_labels_clean.npy", fused_labels)

print(" Final fused shape:", fused_features.shape)
print(" Labels shape:", fused_labels.shape)


In [ ]:
import numpy as np
import pandas as pd

# Load
X = np.load("fused_features_clean.npy")
y = np.load("fused_labels_clean.npy")

# Combine into DataFrame
df = pd.DataFrame(X)
df["label"] = y

# Save to CSV
df.to_csv("fused_embeddings_with_labels.csv", index=False)
print(" Saved: fused_embeddings_with_labels.csv")


In [ ]:
import numpy as np

# Load fused embeddings
fused = np.load("fused_features_clean.npy")  # (84755, 640)

# Split features
image_features = fused[:, :256]   # CNN-based
text_features  = fused[:, 256:]   # Sentence-transformer-based

# Save separately
np.save("image_features_only.npy", image_features)
np.save("text_features_only.npy", text_features)

print(" Saved:")
print("  image_features_only.npy (shape:", image_features.shape, ")")
print("  text_features_only.npy  (shape:", text_features.shape, ")")


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 🔹 Step 1: Load fused features and labels
X = np.load("fused_features_clean.npy")   # Shape: (84755, 640)
y = np.load("fused_labels_clean.npy")     # Shape: (84755,)

# 🔹 Step 2: Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 🔹 Step 3: Define and train MLP
mlp = MLPClassifier(
    hidden_layer_sizes=(512, 256, 64),   # You can adjust the architecture
    activation='relu',
    solver='adam',
    max_iter=50,          # Increase to 100–300 for better results if time allows
    random_state=42,
    verbose=True
)
mlp.fit(X_train, y_train)

# 🔹 Step 4: Predict and evaluate
y_pred = mlp.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"\n MLP Accuracy: {acc:.4f}")
print("\n Classification Report:")
print(classification_report(y_test, y_pred, target_names=["AD", "CN", "MCI"]))

# 🔹 Step 5: Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=["AD", "CN", "MCI"], yticklabels=["AD", "CN", "MCI"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(" Confusion Matrix - MLP (Fused Features)")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Assuming y_test and y_pred are already defined from your previous evaluation

# Define class labels (you can update these if your label encoding is different)
class_names = ['AD', 'CN', 'MCI']

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\n Accuracy: {accuracy:.4f}")

# Precision, Recall, F1-score
print("\n Classification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))
